<a href="https://colab.research.google.com/github/hawaekrami/week1/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hawaekrami/week1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I'm predicting is_declining_label (impressions dropped >20% from Feb to March 2026)
a yes/no question with a real observed outcome, so per the toolkit I'm starting with
Logistic Regression: readable, coefficients are explainable in plain language, and its
predicted probability doubles as a ranking score for precision@K (my lane is a
"which pages first" ranking question, and probability-based ranking is exactly what
that needs). I'll only escalate to Random Forest if Logistic Regression clearly
underfits the pattern.

In [1]:
%pip -q install duckdb scikit-learn
import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
HF_TOKEN = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

FEB = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet')"
MAR = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

# Features: February only (strictly BEFORE the label period)
feb = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS impressions_feb,
        SUM(gsc_clicks) AS clicks_feb,
        ROUND(100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0), 2) AS ctr_feb,
        AVG(gsc_avg_position) AS avg_position_feb
    FROM {FEB}
    GROUP BY content_hash_id, client_hash_id
""").df()

# Label source: March impressions, to compare against February
mar = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions_mar
    FROM {MAR}
    GROUP BY content_hash_id
""").df()

data = feb.merge(mar, on="content_hash_id", how="left")
data["impressions_mar"] = data["impressions_mar"].fillna(0)

# trend_pct, mirroring the starter CSV's own definition (blank/0-prev -> 0, not divide-by-zero)
data["trend_pct"] = 0.0
nonzero_prev = data["impressions_feb"] > 0
data.loc[nonzero_prev, "trend_pct"] = (
    100.0 * (data.loc[nonzero_prev, "impressions_mar"] - data.loc[nonzero_prev, "impressions_feb"])
    / data.loc[nonzero_prev, "impressions_feb"]
)

data["is_declining_label"] = (data["trend_pct"] < -20).astype(int)

print("Total pages:", len(data))
print("Declining rate (base rate):", round(data["is_declining_label"].mean() * 100, 1), "%")
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total pages: 321546
Declining rate (base rate): 14.4 %


,content_hash_id,client_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,impressions_mar,trend_pct,is_declining_label
0,content_7995404695ee1ffd,client_e547b89c05043229,1012.0,3.0,0.30,29.609070,768.0,-24.110672,1
1,content_1eea820697c3b95a,client_e547b89c05043229,299.0,0.0,0.00,12.946228,315.0,5.351171,0
2,content_ccbb253f142217c3,client_e547b89c05043229,1598.0,7.0,0.44,17.806923,3071.0,92.177722,0
3,content_5f58c55cbfee172a,client_e547b89c05043229,514.0,0.0,0.00,10.490023,387.0,-24.708171,1
4,content_6fe390ba3af1e456,client_e547b89c05043229,2931.0,3.0,0.10,38.436254,4697.0,60.252474,0


## 2. Split design

Two honest constraints, combined: (1) Time-aware every feature (impressions_feb,
clicks_feb, ctr_feb, avg_position_feb) is measured in February, strictly before the
label period (Feb-to-March trend), so the model never sees the future it's predicting.
(2) Grouped by client — I split on client_hash_id, not on individual rows, so every
page from a given client lands entirely in train or entirely in test. This matters
because pages from the same client tend to share client-specific baseline behavior
(their SEO team, their content style); a random row split would leak that shared
client pattern between train and test and make the score look better than it really is.

In [2]:
import numpy as np

rng = np.random.default_rng(42)  # fixed seed, so this split is reproducible

clients = data["client_hash_id"].unique()
rng.shuffle(clients)

n_test = int(len(clients) * 0.2)
test_clients = set(clients[:n_test])
train_clients = set(clients[n_test:])

train = data[data["client_hash_id"].isin(train_clients)].copy()
test = data[data["client_hash_id"].isin(test_clients)].copy()

print("Train clients:", len(train_clients), "| Train rows:", len(train))
print("Test clients:", len(test_clients), "| Test rows:", len(test))
print("Train decline rate:", round(train["is_declining_label"].mean() * 100, 1), "%")
print("Test decline rate:", round(test["is_declining_label"].mean() * 100, 1), "%")


Train clients: 44 | Train rows: 277889
Test clients: 10 | Test rows: 43657
Train decline rate: 14.9 %
Test decline rate: 10.8 %


## 3. Train + compare vs my baseline

Same test set, same metric (precision@50), for both: my Week-4 rule (recomputed on
February features, so it's judged on the same information the model gets) and a
Logistic Regression trained on the train clients only.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

features = ["impressions_feb", "clicks_feb", "ctr_feb", "avg_position_feb"]

X_train = train[features].fillna(0)
y_train = train["is_declining_label"]
X_test = test[features].fillna(0)
y_test = test["is_declining_label"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(random_state=42, class_weight="balanced")
model.fit(X_train_scaled, y_train)

test = test.copy()
test["model_score"] = model.predict_proba(X_test_scaled)[:, 1]

# Recompute the Week-4 rule on February features, same eligibility/logic as before
CTR_BENCHMARK = 0.5
eligible = (test["impressions_feb"] >= 500) & (test["avg_position_feb"] > 0) & (test["avg_position_feb"] <= 20)
ctr_gap = (CTR_BENCHMARK - test["ctr_feb"]).clip(lower=0)
test["baseline_score"] = 0.0
test.loc[eligible, "baseline_score"] = test.loc[eligible, "impressions_feb"] * ctr_gap[eligible]

def precision_at_k(df, score_col, k=50):
    top_k = df.sort_values(score_col, ascending=False).head(k)
    return top_k["is_declining_label"].mean() * 100

base_rate = y_test.mean() * 100
baseline_p50 = precision_at_k(test, "baseline_score", 50)
model_p50 = precision_at_k(test, "model_score", 50)

comparison = pd.DataFrame({
    "method": ["Base rate (random)", "Week-4 baseline rule", "Logistic Regression"],
    "precision@50": [round(base_rate, 1), round(baseline_p50, 1), round(model_p50, 1)]
})
print(comparison.to_string(index=False))


              method  precision@50
  Base rate (random)          10.8
Week-4 baseline rule          20.0
 Logistic Regression          40.0


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.